# Text Emotion Recognition — Experiments

Step-by-step fine-tuning of text embeddings for emotion classification using the MELD dataset.
We compare model performance across two input strategies: isolated single utterances vs. context-linked utterances.

| Model | Input Type | Search Space (Hyperparameters) | Epochs |
|---|---|---|---|
| **RoBERTa-base** | Single Utterance (No Context) | Learning Rate `[1e-5, 5e-5]`, Batch Size, Weight Decay | 3-5 |
| **RoBERTa-base** | Linked Utterance (With Context) | Learning Rate `[1e-5, 5e-5]`, Batch Size, Weight Decay | 3-5 |

Run cells top-to-bottom. Models are evaluated using the **weighted F1 score** to account for the heavy class imbalance (majority 'Neutral').

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os

# -------------------------------------------------------
# Setup Paths (Aligning with project structure)
# -------------------------------------------------------
SRC = Path("../src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_ROOT = Path("../data/raw")
MODELS_DIR = Path("../data/models") 

# Ensure the models directory exists so saving doesn't crash later
MODELS_DIR.mkdir(parents=True, exist_ok=True)

EMOTION_ORDER = ["neutral", "joy", "sadness", "anger",
    "surprise", "disgust", "fear"]

EMOTION_COLORS = {"neutral": "#888780", "joy": "#639922", "sadness":  "#378ADD",
    "anger": "#E24B4A", "surprise": "#EF9F27", "disgust":  "#D4537E", "fear": "#7F77DD"}

# -------------------------------------------------------
# Plotting Configuration
# -------------------------------------------------------
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

# -------------------------------------------------------
# Local Hardware Check
# -------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Text Setup done.\n" + "-"*30)
print(f"Hardware computing device: {device}")

# If training locally on a CPU, training will be highly bottlenecked
if device.type == 'cpu':
    print("⚠️ WARNING: No GPU detected. Training RoBERTa locally on a CPU will take a very long time.")
    print("💡 Tip: If it is too slow, swap 'roberta-base' for 'distilroberta-base' in your training code.")
else:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")

print("-" * 30)
print(f"Reading data from: {DATA_ROOT}")
print(f"Saving model weights to: {MODELS_DIR}")

OSError: [WinError 1114] Eine DLL-Initialisierungsroutine ist fehlgeschlagen. Error loading "C:\Users\chris\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
#data processing 
from sklearn.utils.class_weight import compute_class_weight

# 1. Configuration
PROCESSED_ROOT = Path("../data/processed")
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True) 

def process_meld_data(filepath):
    df = pd.read_csv(filepath)
    df.columns = [c.strip() for c in df.columns]
    
    # 2. The Context Logic using the NEW column names
    df['prev_text'] = df.groupby('dialogue_id')['text'].shift(1)
    df['prev_text'] = df['prev_text'].fillna("")
    
    # Create the two versions for your report
    df['text_no_context'] = df['text']
    df['text_with_context'] = df.apply(
        lambda x: f"{x['prev_text']} </s> {x['text']}" if x['prev_text'] != "" 
        else x['text'], axis=1)
    
    # Use the label_idx provided by your team's new structure
    df['label'] = df['label_idx']
    
    return df[['text_no_context', 'text_with_context', 'label', 'emotion', 'dialogue_id', 'utterance_id']]

# 3. Execution (Update these filenames to whatever your team renamed them to)
print("Processing datasets...")
train_df = process_meld_data(DATA_ROOT / 'manifest_train.csv')
val_df   = process_meld_data(DATA_ROOT / 'manifest_dev.csv')  
test_df  = process_meld_data(DATA_ROOT / 'manifest_test.csv')  

# 4. Save Processed Data
print("Saving processed data to ../data/processed/ ...")
train_df.to_csv(PROCESSED_ROOT / 'train_processed.csv', index=False)
val_df.to_csv(PROCESSED_ROOT / 'val_processed.csv', index=False)
test_df.to_csv(PROCESSED_ROOT / 'test_processed.csv', index=False)

# 5. Calculate Weighted Loss Values for the 8 classes
labels = train_df['label'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)

print("\nClass Weights calculated successfully:")
# Dynamically print the weights based on the unique emotions in the train set
unique_emotions = train_df.drop_duplicates('label').sort_values('label')
for _, row in unique_emotions.iterrows():
    print(f"{row['emotion']} (Index {row['label']}): {class_weights[row['label']]:.4f}")

In [ ]:
#viz data balance with new data structure including emotion

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, (df, title) in zip(axes, [
    (train_df, "Train"), (val_df, "Validation"), (test_df, "Test")
]):
    counts = df["emotion"].value_counts().reindex(EMOTION_ORDER).fillna(0)
    colors = [EMOTION_COLORS[e] for e in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=0.8)
    ax.set_title(f"{title} ({len(df):,} Utterances)", fontweight="bold")
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Anzahl Utterances")
    ax.tick_params(axis="x", rotation=40)
    # Werte über den Balken
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 10, f"{int(h):,}",
                ha="center", va="bottom", fontsize=9)

plt.suptitle("Klassenverteilung über alle Splits", fontsize=14, y=1.02)
plt.tight_layout()
os.makedirs("../outputs/plots", exist_ok=True)
plt.savefig("../outputs/plots/class_distribution.png", bbox_inches="tight")
plt.show()
print("Plot gespeichert.")

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
import torch
from torch import nn

# 1. Load the Tokenizer
model_name = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(model_name)

# 2. Define the PyTorch Dataset
class MELDTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 3. Define the Evaluation Metric (Now tracking Macro F1)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    # Calculate both so you have full visibility for your report
    macro_f1 = f1_score(labels, preds, average='macro')
    weighted_f1 = f1_score(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    
    return {
        'macro_f1': macro_f1,          # The optimizer will look at this!
        'weighted_f1': weighted_f1,    # Kept for reference
        'accuracy': acc
    }

In [ ]:
# Convert the class weights we calculated earlier into a PyTorch tensor
# Note: Ensure 'class_weights' from Cell 1 is in memory, or recalculate it here.
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Extract labels
        labels = inputs.pop("labels")
        # Get model predictions
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Calculate loss using our custom weights
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

In [ ]:
def train_and_evaluate(text_column_name, run_name):
    print(f"\n--- Starting Run: {run_name} using column '{text_column_name}' ---")
    
    # 1. Create Datasets
    train_dataset = MELDTextDataset(train_df[text_column_name], train_df['label'], tokenizer)
    val_dataset = MELDTextDataset(val_df[text_column_name], val_df['label'], tokenizer)
    
    # 2. Initialize a fresh model
    model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=8)
    model.to(device)
    
    # 3. Define Training Arguments
    run_output_dir = MODELS_DIR / run_name
    training_args = TrainingArguments(
        output_dir=str(run_output_dir),
        num_train_epochs=4,              
        per_device_train_batch_size=16,  
        per_device_eval_batch_size=32,
        learning_rate=2e-5,              
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,     
        metric_for_best_model="macro_f1", # <--- CHANGED: Now optimizing for Macro F1
    )
    
    # 4. Initialize our Custom Trainer
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )
    
    # 5. Train!
    print("Training started...")
    trainer.train()
    
    # 6. Final Evaluation
    print("\nEvaluating best model on Val Set...")
    eval_results = trainer.evaluate()
    
    # Updated print statement to show the new primary metric
    print(f"Final Val Macro F1: {eval_results['eval_macro_f1']:.4f}")
    print(f"(Reference) Val Weighted F1: {eval_results['eval_weighted_f1']:.4f}")
    
    # Save the final best model
    trainer.save_model(str(run_output_dir / "best_model"))
    print(f"Model saved to {run_output_dir / 'best_model'}")

# ==========================================
# UNCOMMENT TO RUN ON DESKTOP
# ==========================================
# Run 1: Baseline (No Context)
# train_and_evaluate(text_column_name='text_no_context', run_name='roberta_baseline')

# Run 2: With Context
# train_and_evaluate(text_column_name='text_with_context', run_name='roberta_context')